# Day 4.1: MLOps Experiment Tracking

Today we start the MLOps part of the workflow.

The goal of this notebook is not to build the most accurate model immediately. The goal is to understand why model training needs to be tracked in a structured way.

By the end of this notebook, you should be able to answer:

1. What problem does MLOps solve?
2. What are this model's input and output?
3. What are data versions and feature versions?
4. What does a normal Python training script print?
5. What does MLflow record that plain Python does not?
6. How can we compare multiple training runs in the MLflow UI?




## 1. Why MLOps?

A normal machine learning notebook or script can train a model, but it usually leaves many questions unanswered:

- Which dataset did we use?
- Which features did we use?
- Which model settings did we use?
- What were the metrics?
- Where is the saved model?
- If we trained four models, which one was better?
- Can another person reproduce this result later?

MLOps is about making the machine learning workflow more visible, repeatable, and manageable.

First, we focus on one part of MLOps:

```text
Experiment Tracking
```

Experiment tracking means recording what happened during training.




## 2. Today's Experiment Setting

We use the taxi and weather data collected earlier.

The prediction task is:

```text
Use current taxi, time, and weather information
 to predict taxi availability about 2 hours later.
```

One training row represents one planning area at one current timestamp.

The model input describes what we know now.

The model output is what we want to predict about 2 hours later.




## 3. Input And Output

### Output / Target

The target column is:

```text
target_taxi_count_2h
```

Meaning:

```text
Number of available taxis in the same planning area about 2 hours later.
```

### Input Features

The input features can include:

```text
hour
day_of_week
is_weekend
is_rain_forecast_2h
weather_category_id
current_taxi_count
```

The first model will start with only a few simple features. Later we add more features and compare the result.



## 4. Data Versions And Feature Versions

We separate two ideas:

```text
Data version = which dataset file did we train on?
Feature version = which input columns did we use?
```

### Data Versions

```text
data v0 = taxi_weather_2h_prediction_dataset.csv
data v1 = taxi_weather_2h_prediction_dataset_0606.csv
data v2 = taxi_weather_2h_prediction_dataset_0617.csv
```

`data v1` has more collected records than `data v0`.

`data v2` is a larger optional dataset collected over a longer period. The main comparison uses `v0` and `v1`; `v2` is useful for an extension exercise.

### Feature Versions

```text
feature v0 = hour,day_of_week,is_weekend
```

```text
feature v1 = hour,day_of_week,is_weekend,is_rain_forecast_2h,weather_category_id,current_taxi_count
```

This gives us four main runs to compare:

```text
data v0 + feature v0
data v0 + feature v1
data v1 + feature v0
data v1 + feature v1
```

As an extension, try one more production-style question:

```text
What changes when we train the same model on the larger data v2?
```






## 5. How The Feature Table Was Prepared

The prepared training table is stored as CSV files in:

```text
day_4/day_4_mlflow/
```

The main columns are:

| Column | Meaning |
|---|---|
| `timestamp` | current taxi API timestamp |
| `planning_area` | Singapore planning area |
| `hour_of_day` | hour of the current timestamp |
| `day_of_week` | day of week, Monday is `0` |
| `is_weekend` | `1` if Saturday/Sunday, else `0` |
| `forecast_2h` | weather forecast around 2 hours later |
| `is_rain_forecast_2h` | `1` if forecast contains rain/showers/thundery |
| `current_taxi_count` | current taxi count in the area |
| `target_taxi_count_2h` | taxi count about 2 hours later |

The optional script that prepares this table is:

```text
day_4_mlflow/prepare_2h_prediction_dataset.py
```

It takes a Day 1 style SQLite database and builds rows like:

```text
current taxi/weather features at time T
-> taxi count target around T + 2 hours
```

A larger prepared dataset is also available:

```text
taxi_weather_2h_prediction_dataset_0617.csv
```

This is a realistic MLOps pattern: the model code can stay the same while the training data version changes.






In [1]:
import pandas as pd

for path in [
    "day_4_mlflow/taxi_weather_2h_prediction_dataset.csv",
    "day_4_mlflow/taxi_weather_2h_prediction_dataset_0606.csv",
]:
    df = pd.read_csv(path)
    print(path)
    print("rows:", len(df))
    print("columns:", list(df.columns))
    print()

day_4_mlflow/taxi_weather_2h_prediction_dataset.csv
rows: 20796
columns: ['timestamp', 'desired_target_timestamp', 'actual_target_timestamp', 'planning_area', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_peak_hour', 'is_late_night', 'forecast_2h', 'is_rain_forecast_2h', 'current_taxi_count', 'target_taxi_count_2h']

day_4_mlflow/taxi_weather_2h_prediction_dataset_0606.csv
rows: 65374
columns: ['timestamp', 'desired_target_timestamp', 'actual_target_timestamp', 'planning_area', 'hour_of_day', 'day_of_week', 'is_weekend', 'is_peak_hour', 'is_late_night', 'forecast_2h', 'is_rain_forecast_2h', 'current_taxi_count', 'target_taxi_count_2h']



In [3]:
dataset_path = "day_4_mlflow/taxi_weather_2h_prediction_dataset.csv"
df = pd.read_csv(dataset_path)
df.head()

,timestamp,desired_target_timestamp,actual_target_timestamp,planning_area,hour_of_day,day_of_week,is_weekend,is_peak_hour,is_late_night,forecast_2h,is_rain_forecast_2h,current_taxi_count,target_taxi_count_2h
0,2026-05-25 22:49:45+08:00,2026-05-26 00:49:45+08:00,2026-05-26 00:49:50+08:00,ANG MO KIO,22,0,0,0,0,Cloudy,0,67,139
1,2026-05-25 22:49:45+08:00,2026-05-26 00:49:45+08:00,2026-05-26 00:49:50+08:00,BEDOK,22,0,0,0,0,Cloudy,0,151,107
2,2026-05-25 22:49:45+08:00,2026-05-26 00:49:45+08:00,2026-05-26 00:49:50+08:00,BISHAN,22,0,0,0,0,Cloudy,0,59,26
3,2026-05-25 22:49:45+08:00,2026-05-26 00:49:45+08:00,2026-05-26 00:49:50+08:00,BOON LAY,22,0,0,0,0,Cloudy,0,6,2
4,2026-05-25 22:49:45+08:00,2026-05-26 00:49:45+08:00,2026-05-26 00:49:50+08:00,BUKIT BATOK,22,0,0,0,0,Cloudy,0,80,66


## 6. First: Run The Plain Python Training Script

Before adding MLflow, run a normal training script first.

The plain script is:

```text
day_4_mlflow/train_linear_regression.py
```

It does not use MLflow. It only prints results in the terminal.

By default, it uses the same starting setup as the MLflow script:

```text
data version = v0
feature version = v0
```

From the `day_4/day_4_mlflow` directory, run:

```powershell
python train_linear_regression.py
```

Or from the `slides` root directory:

```powershell
cd day_4\day_4_mlflow
python train_linear_regression.py
```

Discussion:

> Where are the results stored?

Expected answer:

```text
Only in the terminal output.
```

This is the limitation that motivates MLflow.





## 7. What The Plain Script Does

The plain script follows this machine learning workflow:

```text
load CSV
-> define X and y
-> train/test split
-> preprocess columns
-> train model
-> calculate MAE, RMSE, R2
-> print results
```

It also prints a prediction preview.

This is useful, but it does not create a structured experiment record.

If we run the script four times, we must manually copy the results somewhere. That is not reliable.



## 8. From Plain Python To MLflow

The MLflow version is:

```text
day_4_mlflow/train_linear_regression_mlflow.py
```

It reuses the same idea, but adds MLflow commands.

The important MLflow additions are:

```python
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("taxi_availability_2h_prediction")
```

```python
with mlflow.start_run(run_name=run_name):
    ...
```

```python
mlflow.log_param("data_version", args.data_version)
mlflow.log_param("feature_version", args.feature_version)
mlflow.log_metric("MAE", mae)
mlflow.log_metric("RMSE", rmse)
mlflow.log_metric("R2", r2)
```

```python
mlflow.data.from_pandas(...)
mlflow.log_input(dataset, context="training")
```

```python
mlflow.sklearn.log_model(pipeline, name="model")
```

The training is still done by our Python code. MLflow records what happened.



## 9. What MLflow Will Record

For each training run, inspect these parts of the MLflow UI:

| UI area | What to look for |
|---|---|
| Run name | Which training attempt is this? |
| Parameters | data version, feature version, model type, feature list |
| Metrics | MAE, RMSE, R2 |
| Inputs / Dataset | dataset name, source, schema, row count |
| Models | logged model from `mlflow.sklearn.log_model` |
| Artifacts | saved `model` folder |

First round questions:

```text
Can you see which dataset was used?
Can you see which features were used?
Can you see model performance?
Can you see that the model was saved?
```




## 10. Start The Unified Day 4 Docker Environment

By Day 4, we use one Docker working folder for MLflow, model serving, and the feature-store demo:

```text
day_4/day_4_mlflow/
```

Open a terminal and start the environment from that folder:

```powershell
cd day_4\day_4_mlflow
docker compose up --build -d
```

The `-d` flag starts the container in the background, so you can keep using the same terminal. The container name is:

```text
day4-mlops
```

The container starts the MLflow Tracking Server automatically.

Open the MLflow UI:

```text
http://localhost:5000
```

Quick check from the same terminal:

```powershell
docker exec -it day4-mlops ls /workspace
```

You should see files such as:

```text
train_linear_regression_mlflow.py
train_mlp_regression_mlflow.py
taxi_weather_2h_prediction_dataset.csv
artifacts
mlflow.db
```

Why this setup is simpler:

| Path | Meaning |
|---|---|
| local `day_4/day_4_mlflow` | the folder you can inspect in VS Code |
| container `/workspace` | the same folder inside Docker; use this for commands |
| container `/mlflow` | compatibility path for older MLflow artifact records |
| `/workspace/mlflow.db` | MLflow metadata: runs, params, metrics, model registry |
| `/workspace/artifacts` | saved model files |

The important idea:

```text
Training script logs runs -> MLflow server stores records -> browser shows experiments
```

When you finish the Day 4 demo, stop the background services from the same folder:

```powershell
docker compose down
```






## 11. Open The MLflow UI

After the server starts, open:

```text
http://localhost:5000
```

At the beginning, the experiment may be empty or may show previous class runs.

Discussion:

> What do you see before running the MLflow script?

This gives them a baseline before the next step.




## 12. Run The MLflow Training Script

Run training inside the same `day4-mlops` container:

```powershell
docker exec -it -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v0 --feature-version v0
```

Meaning:

| Part | Meaning |
|---|---|
| `docker exec -it` | run a command inside the existing container |
| `-w /workspace` | use `/workspace` as the working directory |
| `day4-mlops` | the unified Day 4 container |
| `python train_linear_regression_mlflow.py` | run our training script |
| `--data-version v0` | use the first dataset |
| `--feature-version v0` | use the simple feature set |

The script connects to MLflow at `http://localhost:5000` from inside the container.

The run name will be generated automatically:

```text
linear_regression_data_v0_features_v0
```




## 13. Observe What Changed In MLflow

Refresh:

```text
http://localhost:5000
```

Click the experiment:

```text
taxi_availability_2h_prediction
```

Open the run:

```text
linear_regression_data_v0_features_v0
```

Check:

```text
Parameters
Metrics
Inputs / Dataset
Models
Artifacts
Run name
```

Ask:

> What did MLflow record that the plain Python script did not keep for us?




## 14. Observe Parameters

In **Parameters**, you should see values like:

```text
data_version = v0
feature_version = v0
features = hour,day_of_week,is_weekend
feature_count = 3
model_type = LinearRegression
```

This answers:

```text
What choices did we make before training?
```




## 15. Observe Metrics

In **Metrics**, you should see:

```text
MAE
RMSE
R2
```

These answer:

```text
How well did the model perform?
```

For each row, let:

$$
y_i = \text{actual taxi count 2 hours later}
$$

$$
\hat{y}_i = \text{predicted taxi count 2 hours later}
$$

$$
n = \text{number of test rows}
$$

| Metric | Formula | How to read it |
|---|---|---|
| MAE | $\frac{1}{n}\sum_{i=1}^{n} \|y_i - \hat{y}_i\|$ | Average absolute error. If MAE = 20, the model is wrong by about 20 taxis on average. Lower is better. |
| RMSE | $\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$ | Similar to MAE, but large mistakes are punished more. Lower is better. |
| R2 | $1 - \frac{\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}{\sum_{i=1}^{n}(y_i - \bar{y})^2}$ | Compares the model with a simple baseline that always predicts the average. Closer to 1 is better; 0 means about the same as the average baseline; negative means worse than that baseline. |

For this taxi regression task, MAE is usually the easiest metric to explain to a non-technical audience:

```text
On average, how many taxis is the prediction wrong by?
```

RMSE helps us notice large errors. R2 helps us compare whether the model is meaningfully better than a simple average prediction.





## 16. Observe Dataset Input

In **Inputs** or **Dataset**, you should see the training dataset logged by MLflow.

Example:

```text
taxi_weather_2h_prediction_v0
source = /mlflow/taxi_weather_2h_prediction_dataset.csv
context = training
```

This is different from only logging `data_path` as a parameter.

Dataset tracking helps answer:

```text
Which data did this run use?
What was the schema?
How many rows were in the dataset?
```




## 17. Observe Models And Artifacts

In **Models**, you should see a logged model created by:

```python
mlflow.sklearn.log_model(pipeline, name="model")
```

In **Artifacts**, you should see:

```text
model
```

The saved model folder usually contains files such as:

```text
MLmodel
model.pkl
requirements.txt
python_env.yaml
```

This answers:

```text
Was the trained model saved in a reusable format?
```




## 18. Model Registry: From Logged Model To Served Model

So far, MLflow has logged a model artifact inside one run. That is useful, but it is still tied to a specific experiment run.

For serving, we usually want a stable model name that an application can load.

| Concept | Meaning |
|---|---|
| Logged model artifact | model files saved under one MLflow run |
| Registered model | a named model in MLflow Model Registry |
| Model version | each registered model can have version 1, version 2, version 3, ... |
| Alias | a readable pointer to one version, such as `champion` |

The serving app can load:

```text
models:/taxi_availability_2h_model@champion
```

This means:

```text
Use the registered model named taxi_availability_2h_model.
Use whichever version currently has the champion alias.
```

This is why a dashboard can keep the same code while the active model version changes later.




In [4]:
!docker exec -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v1 --feature-version v1 --register-model --model-name taxi_availability_2h_model --model-alias champion


Taxi Availability Prediction: MLflow Training Script
MLflow Tracking URI: http://localhost:5000
Experiment name: taxi_availability_2h_prediction
Run name: linear_regression_data_v1_features_v1
Data version: v1
Dataset path: /workspace/taxi_weather_2h_prediction_dataset_0606.csv
Rows: 65374
Feature version: v1
Features: hour,day_of_week,is_weekend,is_rain_forecast_2h,weather_category_id,current_taxi_count
Target column: target_taxi_count_2h

Training Linear Regression
------------------------------------------------------------------------

Model Registry
------------------------------------------------------------------------
Registered model name: taxi_availability_2h_model
Registered model version: 11
Alias: champion
Model URI: models:/taxi_availability_2h_model@champion

Final test results
------------------------------------------------------------------------
MAE:  21.426
RMSE: 38.143
R2:   0.736

Logged to MLflow. Open http://localhost:5000 and inspect the run.
🏃 View run linear_

/usr/local/lib/python3.10/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/usr/local/lib/python3.10/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

After running the command, open the MLflow UI and check:

```text
Models -> taxi_availability_2h_model
```

You should see a model version with alias:

```text
champion
```

Day 4.2 will use this registered model URI from the dashboard API.




## 19. Run Four Experiments

Now run all four combinations of data version and feature version.

```powershell
docker exec -it -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v0 --feature-version v0
```

```powershell
docker exec -it -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v0 --feature-version v1
```

```powershell
docker exec -it -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v1 --feature-version v0
```

```powershell
docker exec -it -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v1 --feature-version v1
```

Expected run names:

```text
linear_regression_data_v0_features_v0
linear_regression_data_v0_features_v1
linear_regression_data_v1_features_v0
linear_regression_data_v1_features_v1
```




## 20. Compare The Four Runs

In the MLflow UI, compare the four runs.

Discussion:

1. Which run has the lowest MAE?
2. Which run has the lowest RMSE?
3. Which run has the highest R2?
4. Did adding more features help?
5. Did using the newer dataset change the result?

This is the core idea of Day 4.1:

```text
MLflow lets us compare training attempts without manually copying terminal output.
```




## 21. Discussion: What Is A Run?

A run is one training attempt.

If we change any of these, it should be a new run:

```text
data version
feature version
model type
model parameter
train/test split
```

This is why the four combinations are four separate runs.

Later, when we try MLP or other models, those should also become new runs.



## 22. Common Issues

### Port 5000 is already in use

Check which process is using the port:

```powershell
netstat -ano | findstr :5000
```

Stop the old container if it is still running:

```powershell
docker rm -f day4-mlops
```

Then start again from the unified folder:

```powershell
cd day_4\day_4_mlflow
docker compose up --build -d
```

### Container name already exists

Run:

```powershell
docker rm -f day4-mlops
```

Then start the environment again.

### Dataset or script file not found inside `/workspace`

Inspect what Docker sees:

```powershell
docker exec -it day4-mlops ls /workspace
```

If the folder looks wrong, stop the container and restart Docker Compose from `day_4/day_4_mlflow`.




## 23. Optional Extension: Try A Larger Data Version

The four main runs use `data v0` and `data v1` so the first comparison stays light.

There is also a larger prepared dataset:

```text
day_4_mlflow/taxi_weather_2h_prediction_dataset_0617.csv
```

It is available as `--data-version v2`. Run the same feature version on the larger data:

```powershell
docker exec -it -w /workspace day4-mlops python train_linear_regression_mlflow.py --data-version v2 --feature-version v1
```

Then compare these runs in MLflow:

```text
linear_regression_data_v1_features_v1
linear_regression_data_v2_features_v1
```

Questions:

1. Does the larger dataset improve MAE, RMSE, or R2?
2. Does the dataset row count change in the MLflow dataset/input view?
3. If performance changes, is it because the model changed or because the data changed?
4. Would you automatically promote the `v2` model to `champion`? Why or why not?

This is closer to a real production question: new data arrives, but we still need evidence before changing the served model.






## 24. Optional Extension: Try An MLP Model

If there is time, run the same experiment with a simple MLP model.

MLP means **multi-layer perceptron**. For this notebook, we do not need to focus on deep learning theory. The important idea is:

```text
Same data versions
Same feature versions
Different model type
Same MLflow comparison workflow
```

Run:

```powershell
docker exec -it -w /workspace day4-mlops python train_mlp_regression_mlflow.py --data-version v1 --feature-version v1
```

Optional: change simple MLP parameters and run again:

```powershell
docker exec -it -w /workspace day4-mlops python train_mlp_regression_mlflow.py --data-version v1 --feature-version v1 --hidden-layer-sizes 64,32 --learning-rate-init 0.001 --max-iter 300
```

Then compare in MLflow:

```text
linear_regression_data_v1_features_v1
mlp_regression_data_v1_features_v1
```


The MLP script also logs a step-by-step metric:

```text
training_loss
```

Open the MLP run in MLflow, go to **Metrics**, and click `training_loss` to see the loss curve. The script also logs `loss_curve/training_loss_curve.png` and `loss_curve/training_loss_curve.csv` under **Artifacts**, which is useful if the metric chart is hard to find in the UI.

Questions:

1. Did the MLP improve MAE, RMSE, or R2?
2. What new parameters appear in MLflow?
3. Is a more complex model automatically better?

```text
MLflow lets us compare model types using the same experiment tracking process.
```








## 25. Summary

Today we learned the first practical MLflow skill: experiment tracking.

The workflow was:

```text
Understand MLOps need
-> define input and output
-> define data versions and feature versions
-> run plain Python training
-> start MLflow server with Docker
-> run MLflow training script
-> inspect Parameters, Metrics, Dataset, Models, Artifacts
-> compare four runs
```

Key idea:

```text
MLflow does not replace model training.
MLflow records the training process so we can compare, reproduce, and manage it.
```

Next step:

```text
Model Registry, model loading, model serving, and prediction logging.
```



